# EMA Support/Resistance Scanner

## Tool for Ranking Moving Averages by Market Interaction

## Table of contents

- [Core pipeline: Fetch → Analyze → Run functions](#Core-pipeline:-Fetch-→-Analyze-→-Run-functions)
- [Analysis](#Analysis)
  - [EMA as Support](#EMA-as-Support)
  - [EMA as Resistance](#EMA-as-Resistance)
  - [Universal EMA](#Universal-EMA)
- [Visual analysis](#Visual-analysis)
- [Data export](#Data-export)
- [Conclusion](#Conclusion)

__Goal:__ \
Quantify and compare EMA behaviour to understand which periods price actually respects.

__Data:__ \
ByBit Perpetual Futures.

__Workflow:__ \
For each EMA period in a user-supplied range, count how many candles:
  * had their Low   within delta of the EMA   -> low_touch
  * had their High  within delta of the EMA   -> high_touch
  * had the EMA inside their [Low, High] range  -> cross

__Inputs:__
- symbol
- interval (ByBit codes: 1/3/5/15/30/60/120/240/360/720/D/W/M)
- start/end dates (YYYY-MM-DD)
- ema_range (any iterable of integers, e.g. range(1, 101) or [9, 21, 50, 200])
- delta
- delta_mode

__Delta modes:__
- "percent" — delta is a % of the EMA value at each candle (recommended, scales across price regimes) \
The allowed distance changes with price.
- "absolute" — delta is a fixed price distance in quote currency (e.g. USDT). Distance in price units. \
The allowed distance is always exactly the chosen 'number' regardless of price.

__Metric definitions:__ \
(for every candle, per EMA period)
- low_touch  →  |Low  − EMA| ≤ delta
- high_touch →  |High − EMA| ≤ delta
- cross      →  Low ≤ EMA ≤ High (the candle's range straddles the EMA)

__Notes:__
- The first period candles are skipped per EMA (warm-up); toggle with skip_warmup=False if you don't want that.
- Pagination handles arbitrarily long date ranges (1000-candle chunks).
- Uses ewm(span=period, adjust=False) — standard Wilder/TradingView-style EMA.

## Core pipeline: Fetch → Analyze → Run functions

In [ ]:
import time
import requests
import pandas as pd
import numpy as np
from typing import Iterable
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
BYBIT_API = "https://api.bybit.com/v5/market/kline"

__1. Data fetch__

In [3]:
def fetch_bybit_klines(
    symbol: str,
    interval: str,
    start: str,
    end: str,
    category: str = "linear",       # perpetual futures (USDT-margined)
) -> pd.DataFrame:
    """
    Pull kline data from ByBit v5 with automatic pagination.

    symbol   : e.g. "BTCUSDT"
    interval : "1","3","5","15","30","60","120","240","360","720","D","W","M"
    start/end: ISO date strings, e.g. "2024-01-01"
    category : "linear"  -> USDT perps (default)
               "inverse" -> coin-margined perps
    """
    # Convert dates to milliseconds
    # end date is exclusive
    # to extend to end-of-day (so include the end date): end_ts = int(pd.Timestamp(end, tz="UTC").timestamp() * 1000) + 86_400_000 - 1
    start_ts = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    end_ts   = int(pd.Timestamp(end,   tz="UTC").timestamp() * 1000)

    rows, cursor_end = [], end_ts

    while True:
        params = {
            "category": category,
            "symbol":   symbol.upper(),
            "interval": interval,
            "start":    start_ts,
            "end":      cursor_end,
            "limit":    1000,                 # ByBit max per request
        }
        r = requests.get(BYBIT_API, params=params, timeout=20)
        r.raise_for_status()
        data = r.json()

        if data.get("retCode") != 0:
            raise RuntimeError(f"ByBit API error: {data}")

        batch = data["result"]["list"]        # newest first
        if not batch:
            break

        rows.extend(batch)

        # Result_list is sorted DESCENDING (newest candle first)
        oldest_ts = int(batch[-1][0])
        if oldest_ts <= start_ts or len(batch) < 1000:
            break
        # Move backward to fetch older candles
        cursor_end = oldest_ts - 1
        time.sleep(0.1)                       # polite rate-limiting

    if not rows:
        raise ValueError(
            f"No candles returned for {symbol} {interval} between {start} and {end}"
        )

    # Create DataFrame
    cols = ["timestamp", "open", "high", "low", "close", "volume", "turnover"]
    df = pd.DataFrame(rows, columns=cols)

    # Convert types
    df = df.astype({c: float for c in cols[1:]})
    df["timestamp"] = pd.to_datetime(df["timestamp"].astype(np.int64), unit="ms", utc=True)
    
    # Remove any duplicates and sort ascending by time
    df = (df.drop_duplicates("timestamp")
            .sort_values("timestamp")
            .reset_index(drop=True))
    
    # Filter exact date range (inclusive)
    df = df[(df["timestamp"] >= pd.Timestamp(start, tz="UTC")) &
            (df["timestamp"] <= pd.Timestamp(end,   tz="UTC"))].reset_index(drop=True)
    
    return df


__2. Touch / cross analysis__

Main algorithm:
1. Fetches ByBit perpetual futures data
2. For every EMA period in ema_range: \
   2.1. Computes EMA on close prices \
   2.2. Counts:
      - low_touch: candles where |Low - EMA| <= delta   ("candles' minimum (Low +/- delta) touched it")
      - high_touch: candles where |High - EMA| <= delta  ("candles' maximum (High +/- delta) touched it")
      - cross: candles where Low <= EMA <= High     ("candles crossed it")
    
Returns a clean DataFrame with one row per EMA period.

In [4]:
# skip_warmup=True: ignore the first N candles (where N = the EMA period)
# Example: For EMA 50, it throws away the first 50 candles and only counts touches/cross starting from candle 51.
# Set it to False if you want to include every candle regardless.

# delta %: the allowed distance changes with price
# delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price

def analyze_ema_touches(
    df: pd.DataFrame,
    ema_range: Iterable[int],
    delta: float,
    delta_mode: str = "percent",   # "percent" (of EMA) or "absolute"
    skip_warmup: bool = True,
) -> pd.DataFrame:
    """
    Returns a DataFrame with one row per EMA period:
        ema | low_touch | high_touch | cross | evaluated_candles
    """
    close, high, low = df["close"], df["high"], df["low"]
    results = []

    for period in ema_range:
        # Compute EMA (pandas built-in, adjust=False = classic EMA)
        ema = close.ewm(span=period, adjust=False).mean()

        if delta_mode == "percent":
            tol = ema.abs() * (delta / 100.0)
        elif delta_mode == "absolute":
            tol = pd.Series(delta, index=ema.index)
        else:
            raise ValueError("delta_mode must be 'percent' or 'absolute'")

        # Optionally skip the first `period` candles while the EMA is warming up
        warmup = period if skip_warmup else 0
        mask = pd.Series(False, index=ema.index)
        mask.iloc[warmup:] = True

        # Vectorized counts (very fast)
        low_touch  = ((low  - ema).abs() <= tol) & mask
        high_touch = ((high - ema).abs() <= tol) & mask
        any_touch = (low_touch | high_touch)
        crossed    = (low <= ema) & (ema <= high) & mask
        above = (low > ema) & mask
        below = (high < ema) & mask

        results.append({
            "ema":               period,
            "low_touch":         int(low_touch.sum()),
            "high_touch":        int(high_touch.sum()),
            "any_touch":         int(any_touch.sum()),
            "cross":             int(crossed.sum()),
            "above":             int(above.sum()),
            "below":             int(below.sum()),
            "evaluated_candles": int(mask.sum()),
        })

    return pd.DataFrame(results)

__3. Calling the fetch and analyze functions__

Configuration:
- interval = "60" for 1-hour candles
- ema_range  = range(5, 201, 5) calculates EMA 5, 10, 15, ..., 200
- delta_mode = "percent" or "absolute"
- delta %: the allowed distance between EMA and candle changes with price
- delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price. Distance in price units.

*Examples:*
- delta=0.10 is 0.10 % of the EMA 
- a 5 USDT price difference in % is calculated as:  5 / price (e.g. 67,000) × 100 = 0.00746%
- a 0.0075% delta in USDT is calculated as: price (e.g. 67,000) × 0.0075 / 100 = 5.025 USDT

In [5]:
symbol     = "BTCUSDT"
interval   = "15"
start      = "2026-01-01"
end        = "2026-04-01"
ema_range  = range(1, 200, 1)
delta      = 50
delta_mode = "absolute"

Candles dataframe:

In [6]:
df = fetch_bybit_klines(symbol, interval, start, end)

print(f"Fetching {symbol} interval={interval} from {start} to {end} ...")
print(f"  -> {len(df)} candles downloaded.")

Fetching BTCUSDT interval=15 from 2026-01-01 to 2026-04-01 ...
  -> 8641 candles downloaded.


Dataframe with stats:

In [7]:
result = analyze_ema_touches(df, ema_range, delta, delta_mode)

print(result.to_string(index=False))

 ema  low_touch  high_touch  any_touch  cross  above  below  evaluated_candles
   1       2348        2378       4367   8640      0      0               8640
   2       1582        1578       2716   8570     33     36               8639
   3       1838        1817       3245   8136    269    233               8638
   4       2110        2032       3781   7396    651    590               8637
   5       2187        2171       4052   6615   1042    979               8636
   6       2227        2239       4199   5915   1372   1348               8635
   7       2170        2176       4098   5303   1664   1667               8634
   8       2099        2123       3993   4847   1885   1901               8633
   9       2021        2044       3863   4508   2047   2077               8632
  10       1969        1964       3749   4168   2222   2241               8631
  11       1873        1892       3588   3919   2329   2382               8630
  12       1773        1832       3440   3702   2425

__Alternative: convenience wrapper__

In [8]:
# def run(
#     symbol: str,
#     interval: str,
#     start: str,
#     end: str,
#     ema_range: Iterable[int],
#     delta: float,
#     delta_mode: str = "percent",
# ) -> pd.DataFrame:
#     df = fetch_bybit_klines(symbol, interval, start, end)
#     result = analyze_ema_touches(df, ema_range, delta, delta_mode)

#     print(f"Fetching {symbol} interval={interval} from {start} to {end} ...")
#     print(f"  -> {len(df)} candles downloaded")

#     return df, result

In [9]:
# if __name__ == "__main__":
#     df, result = run(
#         symbol     = "BTCUSDT",
#         interval   = "15",                 # use "60" for 1-hour candles
#         start      = "2026-01-01",
#         end        = "2026-04-01",
#         ema_range  = range(1, 200, 1),     # range(5, 201, 5): EMA 5, 10, 15, ..., 200
#         delta      = 40,
#         delta_mode = "absolute",           # or "percent". delta=0.10 is 0.10 % of the EMA value
#     )
#     print(result.to_string(index=False))

## Analysis

__Which EMA values were touched most often by candle Lows?__

In [10]:
emas_sorted_by_lows = (result[["ema", "low_touch"]]
           .sort_values("low_touch", ascending=False)
           .set_index("ema"))

emas_sorted_by_lows

,low_touch
ema,
1,2348
6,2227
5,2187
7,2170
4,2110
...,...
183,333
181,333
189,331


__Which EMA values were touched most often by candle Highs?__

In [11]:
emas_sorted_by_highs = (result[["ema", "high_touch"]]
           .sort_values("high_touch", ascending=False)
           .set_index("ema"))

emas_sorted_by_highs

,high_touch
ema,
1,2378
6,2239
7,2176
5,2171
8,2123
...,...
195,341
196,339
197,338


__Which EMA values had the most candles floating entirely above them?__

Price spent most of the period trending above this EMA (EMA acted as a floor / bullish regime).

In [12]:
emas_sorted_by_above = (result[["ema", "above"]]
           .sort_values("above", ascending=False)
           .set_index("ema"))

emas_sorted_by_above

,above
ema,
103,3464
104,3460
113,3459
101,3457
105,3457
...,...
5,1042
4,651
3,269


__Which EMA values had the most candles floating entirely below them?__

Price spent most of the period trending below this EMA (EMA acted as a ceiling / bearish regime).

In [13]:
emas_sorted_by_below = (result[["ema", "below"]]
           .sort_values("below", ascending=False)
           .set_index("ema"))

emas_sorted_by_below

,below
ema,
199,4536
198,4531
197,4527
196,4519
195,4515
...,...
5,979
4,590
3,233


### EMA as Support

High ratio = EMA acts as support: price dips down to it, touches it, bounces without getting sliced through.

In [14]:
# replace(0, np.nan) handles division by zero (if an EMA happens to have 0 cross)

ratio_support = result[["ema", "low_touch", "cross"]].copy()
ratio_support["ratio_support"] = ratio_support["low_touch"] / ratio_support["cross"].replace(0, np.nan)
ratio_support = ratio_support.sort_values("ratio_support", ascending=False).set_index("ema")

ratio_support

,low_touch,cross,ratio_support
ema,,,
114,477,920,0.518478
118,469,909,0.515952
113,475,921,0.515744
116,468,908,0.515419
117,470,912,0.515351
...,...,...,...
5,2187,6615,0.330612
4,2110,7396,0.285289
1,2348,8640,0.271759


*Result:*
- With delta 20: ratios around 20.
- With delta 30: ratio 0.336 for EMAs: 116, 115, and ratio 0.327 for EMAs: 117, 113.
- With delta 40: ratio 0.425 for EMAs: 118, 115, and ratio 0.422 for EMAs: 114, 117, 116.
- With delta 50: ratio 0.517 for EMAs: 114, 118, and ratio 0.515 for EMAs: 113, 116, 117.

*Conclusion:*
- Slow EMAs (99, 200) will naturally score highest on this metric because price rarely reaches them at all: so both touches and cross are low, but cross drop faster.
- Ratio ceiling around 0.42 means even the best support EMA only supported ~42% of the time price reached it. \
The other ~58% of cross, price cut through. In a trending market that's normal.

__Broader support ratio variant.__

Treats "price floated above the EMA" as a form of support, not just direct touches.

Captures EMAs acting as a floor during bullish/trending regimes, where price often respects the level without dipping into touch distance.

- Combines direct touches from above (low_touch) with candles that stayed entirely above the EMA (above).
- Unlike ratio_support, this rewards EMAs that acted as a floor during trending stretches where price never dipped close enough to register a touch.
- Useful in strong uptrends where pure touch-based ratios under-count respected levels.

In [15]:
ratio_support_2 = result[["ema", "low_touch", "above", "cross"]].copy()
ratio_support_2["ratio_support_2"] = (ratio_support_2["low_touch"] + ratio_support_2["above"]) / ratio_support_2["cross"].replace(0, np.nan)
ratio_support_2 = ratio_support_2.sort_values("ratio_support_2", ascending=False).set_index("ema")

ratio_support_2

,low_touch,above,cross,ratio_support_2
ema,,,,
199,340,3232,674,5.299703
198,344,3233,679,5.268041
197,344,3229,688,5.193314
196,347,3234,692,5.174855
195,343,3234,697,5.131994
...,...,...,...,...
5,2187,1042,6615,0.488133
4,2110,651,7396,0.373310
1,2348,0,8640,0.271759


__Filtering out tiny sample sizes.__

An EMA with 3 touches and 1 cross has a ratio of 3.0, but it's based on only 3 touches over the whole period — statistically meaningless. \
Other EMAs might have a lower ratio but a lot of touches, which provides a much more reliable signal.

Rule of thumb for a threshold: \
Require at least ~1 touch per week of data, or whatever feels statistically reasonable for your timeframe. \
E.g. 8640 candles of 15-min data = 90 days. MIN_TOUCHES = 30 means roughly one touch every 3 days — a reasonable floor.

In [16]:
# pick a threshold for the minimum sample size that feels right for your dataset

MIN_TOUCHES_SUP = 30

ratio_support_filtered = ratio_support[ratio_support["low_touch"] >= MIN_TOUCHES_SUP]
ratio_support_filtered

,low_touch,cross,ratio_support
ema,,,
114,477,920,0.518478
118,469,909,0.515952
113,475,921,0.515744
116,468,908,0.515419
117,470,912,0.515351
...,...,...,...
5,2187,6615,0.330612
4,2110,7396,0.285289
1,2348,8640,0.271759


__Low touch vs cross by EMA period.__

Visualization of how often each EMA was touched vs crossed by candle Lows, plus their ratio.

Plots low_touch and cross as overlapping bars across all EMA periods, \
with the touch/cross ratio drawn as a red line on a secondary axis. \
Allows to see raw activity and the resistance ratio on one chart.

Result: a dual-axis chart
- bars for the raw counts
- a red line for the ratio

In [17]:
ratio = result["low_touch"] / result["cross"].replace(0, np.nan)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=result["ema"], y=result["cross"], name="cross",
           marker_color="steelblue", opacity=0.4),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=result["ema"], y=result["low_touch"], name="low_touch",
           marker_color="orange", opacity=0.7),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=result["ema"], y=ratio, name="ratio",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.3f}"),
    secondary_y=True,
)

fig.update_layout(
    title="Low touches vs cross by EMA period",
    barmode="overlay",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Ratio", secondary_y=True, color="red")

fig.show()

Use it for:
- Where the ratio peaks (the red line reaches its highest points on the chart) across the EMA spectrum —> EMAs periods that acted as the best support.
- Are they isolated spikes or broad hills (a whole zone respects support)?
- Whether high-ratio EMAs are backed by real volume of touches, or just low cross (thin bars under a high red line = suspicious).
- Trend in the counts — do touches/cross fall off smoothly as EMA period grows, or are there anomalies?

### EMA as Resistance

High ratio = EMA acts as resistance: price rallies up into it, the candle's High kisses it (within delta), and then price rejects back down without the candle slicing through.

Common expectations:

__On a 15-min chart__: \
Slow EMAs like 99, 144 and 200 often score high on ratio because price rarely reaches them, and when it does it often rejects cleanly.

__In a downtrending or ranging market:__ \
The classic "respected resistance" EMAs tend to rise to the top:
- EMA 20 — short-term resistance, widely watched by intraday traders
- EMA 50 — medium-term, very popular on daily charts, often respected on lower timeframes too
- EMA 200 — long-term, the "institutional" line, strong resistance in downtrends

__In a strong uptrend:__ \
Resistance EMAs barely exist — price stays above most EMAs, so high_touch will be low across the board and the ratio becomes noisy. \
The top of ranked might be dominated by random slow EMAs with tiny sample sizes.

__Adjacent EMAs clustering with nearly identical ratios:__ \
It is the signature of a real resistance zone. Example:
- EMA 47–50 zone — 694–729 touches (huge sample), ratio ~0.46
- EMA 85–87 zone — 487–492 touches, ratio ~0.46 \
-> Price on the chart was genuinely respecting levels near the 50-EMA and near the 86-EMA.

In [18]:
ratio_resistance = result[["ema", "high_touch", "cross"]].copy()
ratio_resistance["ratio_resistance"] = ratio_resistance["high_touch"] / ratio_resistance["cross"].replace(0, np.nan)
ratio_resistance = ratio_resistance.sort_values("ratio_resistance", ascending=False).set_index("ema")

ratio_resistance

,high_touch,cross,ratio_resistance
ema,,,
77,642,1136,0.565141
76,644,1140,0.564912
47,898,1594,0.563363
50,849,1509,0.562624
49,864,1539,0.561404
...,...,...,...
5,2171,6615,0.328193
1,2378,8640,0.275231
4,2032,7396,0.274743


*Result:*
- With delta 20: ratios around 20.
- With delta 30: ratio 0.348 for EMAs: 47, 85, 84, 87.
- With delta 40: ratio 0.462 for EMAs: 86, 87, and ratio 0.459 for EMAs: 50, 85, 47.
- With delta 50: ratio 0.565 for EMAs: 77, 76, and ratio 0.562 for EMAs: 47, 50, 49.

*Conclusion:*
- Two distinct resistance clusters emerged: EMA 47–50 zone and EMA 85–87 zone.
- Sample sizes are strong. Even the 85–87 cluster has ~490 touches.
- The 50 EMA showing up is classic. It's the most-watched medium-term EMA on any timeframe. \
The data shows it worked as resistance during the selected time window and the price showed genuine mean-reversion behavior.
- Fast EMAs (1–5) at the bottom are expected. \
They hug price too tightly, so cross dominate (8640 cross for EMA 1 = every single candle). \
Low ratio = they don't act as resistance, they are price.
- Ratio ceiling around 0.46 means even the best resistance EMA only rejected ~46% of the time price reached it. \
The other ~54% of cross, price cut through. In a trending market that's normal.

__Broader resistance ratio variant.__

Treats "price floated below the EMA" as a form of resistance, not just direct touches.

Captures EMAs acting as a ceiling during bearish/trending regimes, where price often respects the level without rallying into touch distance.

- Combines direct touches from below (high_touch) with candles that stayed entirely below the EMA (below).
- Unlike ratio_resistance, this rewards EMAs that acted as a ceiling during trending stretches where price never rallied close enough to register a touch.
- Useful in strong downtrends where pure touch-based ratios under-count respected levels.

In [19]:
ratio_resistance_2 = result[["ema", "high_touch", "below", "cross"]].copy()
ratio_resistance_2["ratio_resistance_2"] = (ratio_resistance_2["high_touch"] + ratio_resistance_2["below"]) / ratio_resistance_2["cross"].replace(0, np.nan)
ratio_resistance_2 = ratio_resistance_2.sort_values("ratio_resistance_2", ascending=False).set_index("ema")

ratio_resistance_2

,high_touch,below,cross,ratio_resistance_2
ema,,,,
199,322,4536,674,7.207715
198,330,4531,679,7.159057
197,338,4527,688,7.071221
196,339,4519,692,7.020231
195,341,4515,697,6.967001
...,...,...,...,...
5,2171,979,6615,0.476190
4,2032,590,7396,0.354516
1,2378,0,8640,0.275231


__Filtering out tiny sample sizes.__

In [20]:
# pick a threshold for the minimum sample size that feels right for your dataset

MIN_TOUCHES_RES = 30

ratio_resistance_filtered = ratio_resistance[ratio_resistance["high_touch"] >= MIN_TOUCHES_RES]
ratio_resistance_filtered

,high_touch,cross,ratio_resistance
ema,,,
77,642,1136,0.565141
76,644,1140,0.564912
47,898,1594,0.563363
50,849,1509,0.562624
49,864,1539,0.561404
...,...,...,...
5,2171,6615,0.328193
1,2378,8640,0.275231
4,2032,7396,0.274743


__High touch vs cross by EMA period.__

Visualization of how often each EMA was touched vs crossed by candle Highs, plus their ratio.

Plots high_touch and cross as overlapping bars across all EMA periods, \
with the touch/cross ratio drawn as a red line on a secondary axis. \
Allows to see raw activity and the resistance ratio on one chart.

Result: a dual-axis chart
- bars for the raw counts
- a red line for the ratio

In [21]:
ratio = result["high_touch"] / result["cross"].replace(0, np.nan)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=result["ema"], y=result["cross"], name="cross",
           marker_color="steelblue", opacity=0.4),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=result["ema"], y=result["high_touch"], name="high_touch",
           marker_color="orange", opacity=0.7),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=result["ema"], y=ratio, name="ratio",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.3f}"),
    secondary_y=True,
)

fig.update_layout(
    title="High touches vs cross by EMA period",
    barmode="overlay",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Ratio", secondary_y=True, color="red")

fig.show()

Use it for:
- Where the ratio peaks (the red line reaches its highest points on the chart) across the EMA spectrum —> EMAs periods that acted as the best resistance.
- Are they isolated spikes or broad hills (a whole zone respects resistance)?
- Whether high ratios are backed by real volume of touches, or just low cross (thin bars under a high red line = suspicious).
- Trend in the counts — do touches/cross fall off smoothly as EMA period grows, or are there anomalies?

### Universal EMA

EMA as any kind of S/R.

In [22]:
# Combining both touches to analyze EMA as any kind of S/R
ratio_universal = result[["ema", "any_touch", "cross"]].copy()
ratio_universal["ratio_universal"] = ratio_universal["any_touch"] / ratio_universal["cross"].replace(0, np.nan)
ratio_universal = ratio_universal.sort_values("ratio_universal", ascending=False).set_index("ema")

ratio_universal.head(10)

,any_touch,cross,ratio_universal
ema,,,
50,1518,1509,1.005964
53,1432,1430,1.001399
55,1393,1392,1.000718
49,1540,1539,1.000650
52,1462,1462,1.000000
56,1380,1380,1.000000
19,2746,2756,0.996372
51,1490,1496,0.995989
20,2664,2677,0.995144


*Result:*
- With delta 20: ratios around 30.
- With delta 30: ratio 0.63 for EMAs: 53, 116, 87, 18.
- With delta 40: ratio 0.83 for EMAs: 18, 50, and ratio 0.82 for EMAs: 19, 20.
- With delta 50: ratio 1 for EMAs: 50, 53, 55, 49, 52, 56.

__Filtering out tiny sample sizes.__

In [23]:
# pick a threshold for the minimum sample size that feels right for your dataset

MIN_TOUCHES_UNI = 60

ratio_universal_filtered = ratio_universal[ratio_universal["any_touch"] >= MIN_TOUCHES_UNI]
ratio_universal_filtered

,any_touch,cross,ratio_universal
ema,,,
50,1518,1509,1.005964
53,1432,1430,1.001399
55,1393,1392,1.000718
49,1540,1539,1.000650
52,1462,1462,1.000000
...,...,...,...
5,4052,6615,0.612547
4,3781,7396,0.511222
1,4367,8640,0.505440


__Any touch vs cross by EMA period.__

In [24]:
ratio = result["any_touch"] / result["cross"].replace(0, np.nan)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=result["ema"], y=result["cross"], name="cross",
           marker_color="steelblue", opacity=0.4),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=result["ema"], y=result["any_touch"], name="any_touch",
           marker_color="orange", opacity=0.7),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=result["ema"], y=ratio, name="ratio",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.3f}"),
    secondary_y=True,
)

fig.update_layout(
    title="Any touch vs cross by EMA period",
    barmode="overlay",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Ratio", secondary_y=True, color="red")

fig.show()


What you see on the chart:
- Fast EMAs (left side) \
Huge blue bars (crosses), smaller orange bars (touches), low red line (ratio). \
EMA hugs price, gets sliced constantly.
- Medium EMAs (middle) \
Blue and orange come closer together, red line rises. This is the sweet spot.
- Slow EMAs (right side) \
Both bars shrink (price rarely reaches them), red line may spike due to low denominators. Watch for noise here.

The red line's peak is where the best S/R candidates are.

## Visual analysis

An interactive financial chart in Plotly:
- Overlays EMAs on a candlestick chart to see if they line up with swing highs and lows.
- Chart opens showing the last 3 days.
- Drag the range slider or click 1d/1w/1m/3m buttons to jump to different windows.
- After jumping, click the "Autoscale" button in the top-right toolbar to rescale y-axis. Double-clicking the y-axis also works.

Change pd.Timedelta(days=3) if you want a different default window:
- days=1 for tighter zoom
- days=7 for a weekly view

__Configurable parameters:__

In [25]:
symbol                                           # label for the candlestick trace and title, configured at data loading
ema_periods     = [50, 77, 114]                  # a list with any number of EMA periods
ema_colors      = ["orange", "blue", "purple"]   # one color per period
initial_window  = pd.Timedelta(days=90)           # default zoom: how much history to show on load
chart_height    = 900

In [26]:
# Compute EMAs dynamically
for period in ema_periods:
    df[f"ema_{period}"] = df["close"].ewm(span=period, adjust=False).mean()

fig = go.Figure()

# Price candles
fig.add_trace(go.Candlestick(
    x=df["timestamp"],
    open=df["open"], high=df["high"], low=df["low"], close=df["close"],
    name=symbol,
))

# EMA lines
for period, color in zip(ema_periods, ema_colors):
    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df[f"ema_{period}"],
        line=dict(color=color, width=1.5),
        name=f"EMA {period}",
    ))

# Default view: last N days
initial_start = df["timestamp"].iloc[-1] - initial_window
initial_end   = df["timestamp"].iloc[-1]

# Title built from config
ema_label = " & ".join(f"EMA {p}" for p in ema_periods)
chart_title = f"{symbol} — {ema_label}"

fig.update_layout(
    title=chart_title,
    height=chart_height,
    template="plotly_white",
    xaxis=dict(
        range=[initial_start, initial_end],
        rangeslider=dict(visible=True, thickness=0.04),
        rangeselector=dict(
            buttons=[
                dict(count=1, label="1d", step="day",   stepmode="backward"),
                dict(count=7, label="1w", step="day",   stepmode="backward"),
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=3, label="3m", step="month", stepmode="backward"),
                dict(step="all", label="All"),
            ]
        ),
    ),
    yaxis=dict(autorange=True, fixedrange=False),
)

fig.show()

## Data export

Saving dataframes to CSV for further analysis in Excel or Tableau.

In [27]:

filename_df = f"df_{symbol}_{interval}_{start}_{end}.csv"
df.to_csv(filename_df, index=False)
print(f"\nResults saved to {filename_df}")


Results saved to df_BTCUSDT_15_2026-01-01_2026-04-01.csv


In [28]:
filename_ema = f"ema_analysis_{symbol}_{interval}_{start}_{end}_{ema_range}_{delta}.csv"
result.to_csv(filename_ema, index=False)
print(f"\nResults saved to {filename_ema}")


Results saved to ema_analysis_BTCUSDT_15_2026-01-01_2026-04-01_range(1, 200)_50.csv


## Conclusion

__EMA choice:__
- The strongest EMA candidates should have the best mix of ratio and sample size.
- EMAs with slightly higher ratio but fewer touches can acts as a secondary/ slower level.

__EMA confirmation:__
- Overlay chosen EMAs on the interactive financial chart to see if they line up with swing highs and lows.

__Further actions:__
- To use the analysis results in TradingView: 
    - find interesting EMAs here
    - paste specific EMA periods you found into a Pine indicator (e.g. ta.ema(close, 50))
    - plot these periods in Pine Script on the chart.